In [3]:
import pandas as pd 
keep_cols = [ 
    'v024', 'sdist', 'v012', 's116', 'v130', 'v025', 'v103',               # Group 1
    'v149', 'v155', 'v190', 'v717',                                       # Group 2
    'v501', 'v743a', 'v743f', 'v739',                                     # Group 3
    'v157', 'v158', 'v159', 'v171a',                                      # Group 4
    'v481a', 'v481b', 'v481c', 'v481d', 'v481e', 'v481f', 'v481g', 'v481h', 'v481x', # Group 5
    'v467a', 'v467b', 'v467c', 'v467d', 'v467e', 'v467f', 'v467g', 'v467h', 'v467i'  # Group 6
    ] 

chunks = pd.read_stata( 'IAIR7EDT/IAIR7EFL.DTA', columns=keep_cols, chunksize=50000, convert_categoricals=False ) 
out_file = "india_nfhs5.csv" 
first = True 
chunk_count = 0 
total_rows = 0 
saved_rows = 0 
print("Starting processing...") 
for chunk in chunks: 
    chunk_count += 1 
    total_rows += len(chunk) 
    rows_found = len(chunk)
    saved_rows += rows_found
    chunk.to_csv(out_file, mode='w' if first else 'a', header=first, index=False)
    first = False

print(f"Saved {saved_rows} rows to {out_file}") 
print("Processing complete") 
print(f"Total rows scanned: {total_rows}") 
print(f"Total India rows saved: {saved_rows}") 

Starting processing...
Saved 724115 rows to india_nfhs5.csv
Processing complete
Total rows scanned: 724115
Total India rows saved: 724115


In [4]:
df_check = pd.read_csv("india_nfhs5.csv")
print(len(df_check))
print(df_check['v024'].unique())

724115
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 27 28 29 30 31 32 33 34 35 36 37]


In [6]:
import pandas as pd
import numpy as np

# Load the cleaned Rajasthan data
df = pd.read_csv('india_nfhs5.csv')

# Handle missing values: replace 7,8,9 (and 96-99 if present) with NaN
cols_to_null = [c for c in df.columns if c not in ['v012', 'v190', 'sdist', 'v024']]
df[cols_to_null] = df[cols_to_null].replace([7, 8, 9, 96, 97, 98, 99], np.nan)

# Drop columns that are entirely empty
df.dropna(axis=1, how='all', inplace=True)

# Regroup insurance variables
df['govt_insurance'] = df[['v481a','v481b','v481c','v481d','v481e']].apply(
    lambda x: 1 if (x==1).any() else (np.nan if x.isna().all() else 0), axis=1)

df['employer_insurance'] = df[['v481f','v481g']].apply(
    lambda x: 1 if (x==1).any() else (np.nan if x.isna().all() else 0), axis=1)

df['private_insurance'] = df[['v481h','v481x']].apply(
    lambda x: 1 if (x==1).any() else (np.nan if x.isna().all() else 0), axis=1)

# Rename columns
rename_map = {
    'v024':'state','sdist':'district','v012':'age','s116':'caste',
    'v130':'religion','v025':'residence_type','v103':'childhood_residence',
    'v149':'education_level','v155':'literacy','v190':'wealth_index','v717':'occupation',
    'v501':'marital_status','v743a':'decide_health','v743f':'decide_husband_money','v739':'decide_own_money',
    'v157':'freq_newspaper','v158':'freq_radio','v159':'freq_tv','v171a':'internet_use',
    'v467b':'barrier_permission','v467c':'barrier_money',
    'v467d':'barrier_distance','v467e':'barrier_transport','v467f':'barrier_alone',
    'v467g':'barrier_female_dr','v467h':'barrier_no_provider','v467i':'barrier_no_drugs'
}
df.rename(columns=rename_map, inplace=True)

# Keep final columns + insurance aggregates
final_cols = list(rename_map.values()) + ['govt_insurance','employer_insurance','private_insurance']
df = df[[c for c in final_cols if c in df.columns]]

# Convert all columns to integer where possible (keep NaN if present)
for col in df.columns:
    if df[col].dtype == 'float64' and df[col].isna().sum() < len(df):
        df[col] = df[col].astype('Int64')

# Save final cleaned file
df.to_csv('india_nfhs5_cleaned.csv', index=False)
print(f"Final cleaned data saved: {len(df)} rows, {len(df.columns)} columns")

Final cleaned data saved: 724115 rows, 29 columns


In [7]:
import pandas as pd

df = pd.read_csv('india_nfhs5_cleaned.csv')
df.count()

state                   724115
district                724115
age                     724115
caste                   685424
religion                715045
residence_type          724115
education_level         724115
literacy                724115
wealth_index            724115
occupation              100783
marital_status          724115
decide_health            76910
decide_husband_money     75092
decide_own_money         20764
freq_newspaper          724115
freq_radio              724115
freq_tv                 724115
internet_use            108785
barrier_permission      724115
barrier_money           724115
barrier_distance        724115
barrier_transport       724115
barrier_alone           724115
barrier_female_dr       724115
barrier_no_provider     724115
barrier_no_drugs        724115
govt_insurance          724115
employer_insurance      724115
private_insurance       724115
dtype: int64